# S01 — AWS CLI Setup on Ubuntu

This notebook installs AWS CLI v2 on Ubuntu, configures authentication, sets `us-east-1` as the only region, verifies the active identity, and works with Amazon S3 buckets.

> **Security:** Never paste an AWS secret access key into a notebook cell, source file, chat, or Git repository. Run `aws configure` in a terminal and enter credentials only at its hidden prompts. Prefer temporary credentials/SSO for production use; access keys are used here because they are the preferred course workflow.

## 1. Prerequisites

You need a 64-bit x86 Ubuntu machine, `sudo` access, an AWS account, and permission to use STS and S3. All commands below use Ubuntu's Bash shell and `apt`.

In [ ]:
%%bash
set -euo pipefail
uname -a
uname -m

## 2. Install AWS CLI v2

The following installs the official AWS CLI v2 bundle on 64-bit x86 Ubuntu.

In [ ]:
%%bash
set -euo pipefail
sudo apt-get update
sudo apt-get install -y curl unzip
cd /tmp
curl -fsSL "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o awscliv2.zip
unzip -q -o awscliv2.zip
sudo ./aws/install --update
aws --version

Restart the notebook kernel or open a new Ubuntu terminal if `aws` is not immediately found after installation.

## 3. Authentication option A — access key (preferred)

In the AWS console, create or select an IAM user with only the permissions required for this exercise. Create an access key only if your organization permits long-lived keys. Download or copy the secret once and store it in a password manager.

Run the next command **in a Linux terminal**, not as a notebook cell, so credentials do not become notebook output or history:

```bash
aws configure --profile training
```

Enter the access key ID, secret access key, default region `us-east-1`, and output format `json`. This writes credentials to `~/.aws/credentials` and settings to `~/.aws/config`. Do not commit either file.

### Change an existing access key and secret key

If the `training` profile already exists and you need to replace its access key ID and secret access key, rerun this command in an Ubuntu terminal:

```bash
aws configure --profile training
```

Enter the new access key ID and new secret access key at the prompts. Enter `us-east-1` for the region and `json` for the output format. This overwrites the existing credentials for only the `training` profile. Never type the actual key or secret directly into a notebook cell.

In [ ]:
%%bash
# Inspect profile names and non-secret configuration only.
aws configure list-profiles
aws configure list --profile training

### Optional: temporary session credentials

If AWS gives you a three-part temporary credential, configure the access key and secret through `aws configure`, then add the session token from a terminal:

```bash
aws configure set aws_session_token 'YOUR_SESSION_TOKEN' --profile training
```

Do not place the real token in this notebook. Temporary credentials expire and must then be refreshed.

## 4. Authentication option B — browser login with IAM Identity Center (SSO)

Use this when your organization provides an AWS access portal URL and SSO region. Configure once, then log in through the browser:

```bash
aws configure sso --profile training-sso
aws sso login --profile training-sso
```

If the machine has no browser, use `aws sso login --profile training-sso --use-device-code` and open the displayed URL on another device. To end the cached SSO sessions, run `aws sso logout`.

## 5. Select a profile and set the region

This notebook uses the access-key profile named `training`. Exporting `AWS_PROFILE` makes subsequent CLI commands use it. Environment variables apply only to the current shell/cell process, so the examples also pass `--profile` explicitly when clarity matters.

In [ ]:
%%bash
set -euo pipefail
PROFILE=training
REGION=us-east-1
aws configure set region "$REGION" --profile "$PROFILE"
aws configure set output json --profile "$PROFILE"
aws configure get region --profile "$PROFILE"

## 6. Verify authentication

Always verify the account and principal before creating resources. This helps prevent changes in the wrong AWS account.

In [ ]:
%%bash
set -euo pipefail
aws sts get-caller-identity --profile training
aws configure list --profile training

## 7. List S3 buckets

Use either the high-level `aws s3` command or the `aws s3api` command to list the buckets available to the `training` profile.

In [ ]:
%%bash
# List all buckets in the account.
aws s3 ls --profile training

# Return selected bucket fields as a table.
aws s3api list-buckets \
  --profile training \
  --query 'Buckets[].{Name:Name,Created:CreationDate}' \
  --output table

## Troubleshooting

- **Unable to locate credentials:** run `aws configure --profile training`, or log in again with `aws sso login --profile training-sso`.
- **ExpiredToken:** refresh temporary credentials or the SSO session.
- **AccessDenied:** the active IAM principal lacks the required action; inspect it with `aws sts get-caller-identity`.
- **Clock skew/signature error:** enable Linux time synchronization, for example `sudo timedatectl set-ntp true`.